In [0]:
%pip install shap
%pip install xgboost

# Feature eng

In [0]:
# carregando tabela de vendas
df_sales = spark.table("adventure_works.marts.fct_sales_for_ds_analysis")

# exibe amostra
df_sales.show(10)


In [0]:
# carregando tabela de endereço
df_addres = spark.table("adventure_works.marts.dim_address")\
    .select("address_id",
            "country_name",
            "city",
            "country_region_name")

# exibe amostra
df_addres.show(3)

In [0]:
df_merge = df_sales.join(df_addres, on="address_id", how="left")
df_merge = df_merge.orderBy("date_order_date")
df_merge.show(15)

In [0]:
# carregando tabela de endereço
df_dates = spark.table("adventure_works.marts.dim_dates")\
    .select("date_day",
            "day_of_month",
            "day_of_week",
            "day_of_week_name",
            "day_of_year",
            "quarter_of_year",
            "year_number",
            "month_of_year"
            )

# exibe amostra
df_dates.show(3)

In [0]:
df_merge = df_merge.join(df_dates, df_merge["date_order_date"] == df_dates["date_day"], how="right")
# \.drop(df_dates["date_day"])

df_merge = df_merge.orderBy("date_order_date")

df_merge.show(15)

In [0]:
from pyspark.sql import functions as F

result = df_merge.agg(
    F.min("date_order_date").alias("min_date"),
    F.max("date_order_date").alias("max_date")
).collect()[0]

min_date = result["min_date"]
max_date = result["max_date"]

In [0]:
df_merge = df_merge.filter(
    (F.col("date_order_date") >= F.lit(min_date)) &
    (F.col("date_order_date") <= F.lit(max_date))
)

In [0]:
min_date

In [0]:
max_date

In [0]:
df_merge.show(15)

In [0]:
print(df_merge.count())

In [0]:
cols = df_merge.columns
duplicates = [col for col in set(cols) if cols.count(col) > 1]
print(duplicates)  # lista de colunas duplicadas

for col in duplicates:
    # drop a segunda e demais ocorrências
    # Spark não permite manter "uma das iguais" por índice, então geralmente droppa todas da dimensão
    df_merge = df_merge.drop(df_merge[col])

In [0]:
columns = ['address_id','date_order_date']

for col in columns:
    df_merge = df_merge.drop(df_merge[col])

In [0]:
for col in df_merge.columns:
    df_merge = df_merge.withColumn(col, F.col(col).cast("string"))

In [0]:
(
    df_merge.write
      .format("delta")
      .mode("overwrite")  # overwrite, append, etc.
      .saveAsTable("adventure_works.data_science.fact_sales_day_with_calendar")
)

In [0]:
# DataFrame 1: category = 'A'
df_merge_sc = df_merge.filter(df_merge["person_type"] == "SC")

# DataFrame 2: category != 'A'
df_merge_in = df_merge.filter(df_merge["person_type"] == "IN")

In [0]:
df_merge_sc.show(15)

In [0]:
(
    df_merge_sc.drop(df_merge_sc["person_type"]).write
      .format("delta")
      .mode("overwrite")  # overwrite, append, etc.
      .saveAsTable("adventure_works.data_science.fact_sales_SC_day_with_calendar")
)


(
    df_merge_in.drop(df_merge_in["person_type"]).write
      .format("delta")
      .mode("overwrite")  # overwrite, append, etc.
      .saveAsTable("adventure_works.data_science.fact_sales_IN_day_with_calendar")
)

In [0]:
pdf = df_merge_in.toPandas()


In [0]:
pdf.to_csv("adventure_works.data_science.table_output.fact_sales_IN_day_with_calendar.csv", index=False)

finaliza o tratamento da tabela aqui


In [0]:
columns = [
            "date_order_date",
            "product_subcategory_name",
            "product_qty_day",
            "person_type",
            "person_type_name",
            "store_name",
            "city",
            "country_name",
            "country_region_name"]
